In [1]:
from pathlib import Path
import ast

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


ROOT = Path.cwd().parent

FEATURE_FILE = ROOT / "data" / "features" / "movies_features.csv"

df = pd.read_csv(FEATURE_FILE)

In [2]:
import re

df["overview"] = df["overview"].fillna("")
df["overview"] = df["overview"].str.lower()
df["overview"] = df["overview"].apply(
    lambda x: re.sub(r"[^a-z0-9\s]", " ", x)
)
df["overview"] = df["overview"].str.replace(r"\s+", " ", regex=True).str.strip()

In [3]:
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(df["overview"])

In [4]:
X_tfidf.shape

(2963, 14648)

In [5]:
for n in [1000, 3000, 5000]:
    vectorizer = TfidfVectorizer(max_features=n)
    X = vectorizer.fit_transform(df["overview"])
    print(f"max_features={n}: shape={X.shape}")

max_features=1000: shape=(2963, 1000)
max_features=3000: shape=(2963, 3000)
max_features=5000: shape=(2963, 5000)


In [6]:
for ngram in [(1,1), (1,2), (1,3)]:
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=ngram)
    X = vectorizer.fit_transform(df["overview"])
    print(f"ngram_range={ngram}: shape={X.shape}")

ngram_range=(1, 1): shape=(2963, 5000)
ngram_range=(1, 2): shape=(2963, 5000)
ngram_range=(1, 3): shape=(2963, 5000)


In [7]:
import joblib
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words="english"
)

X_tfidf = vectorizer.fit_transform(df["overview"])
terms = vectorizer.get_feature_names_out()
mean_scores = X_tfidf.mean(axis=0).A1
top_indeces = mean_scores.argsort()[-20:][::-1]


joblib.dump(vectorizer, ROOT / "models/tfidf_vectorizer.joblib")
joblib.dump(X_tfidf, ROOT / "models/tfidf_matrix.joblib")


['/home/elgmouri/projects/moviesHub/models/tfidf_matrix.joblib']

In [10]:
terms

array(['000', '000 years', '007', ..., 'zombies', 'zone', 'zoo'],
      shape=(5000,), dtype=object)